# **Feature Engineering**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

# Scalling untuk MinMax Scaller
from sklearn.preprocessing import MinMaxScaler
# Set the maximum number of columns and rows to display to a large number
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# **Fitur Handling Duplikat**

1. Saat kita ingin melihat ada atau tidaknya duplikat pada data, kita bisa melihat panjang baris dari data yang kita punya

In [ ]:
df = pd.read_excel('titanic.xlsx')

In [ ]:
len(df) #mengecek panjang baris dari df

In [ ]:
len(df) - len(df.drop_duplicates()) #mengecek selisih baris dari df dengan df yg telah dicoba drop duplikat

Pada kode (len(df) - len(df.drop_duplicates())), kita menghitung jumlah baris yang duplikat dengan "*menghitung seluruh jumlah baris dalam DataFrame (len(df)) dan menguranginya dengan jumlah baris setelah menghapus duplikat*" menggunakan metode drop_duplicates.

Ini menghitung semua duplikat di DataFrame.

In [ ]:
len(df.drop_duplicates()) #mengecek panjang baris ketika sudah diperlakukan drop duplikat

kita juga bisa melihat baris yang duplikat

In [ ]:
len(df.drop_duplicates()) / len(df) #jika output dari code di cell ini tidak bernilai 1 maka terdapat duplikat

In [ ]:
# Menampilkan baris yang memiliki duplikat berdasarkan semua kolom, nanti hanya pilih 1 saja, karena yang duplikat hanya 44 baris
duplicates = df[df.duplicated(keep=False)]

print("Baris dengan duplikat:")
duplicates

In [ ]:
df = df.drop_duplicates() #drop duplikat

In [ ]:
#mengecek selisih baris dari df dengan df yg telah dicoba drop duplikat
len(df) - len(df.drop_duplicates())

In [ ]:
len(df.drop_duplicates()) / len(df)
#jika output dari code di cell ini tidak bernilai 1 maka terdapat duplikat

kita sudah handling drop duplikat

Drop duplikat **SELESAI**

# **Outlier Handling**

Outlier adalah data yang sangat aneh dibandingkan dengan data lainnya, hingga menimbulkan kecurigaan bahwa data tersebut berasal dari sumber yang berbeda. Hal ini dapat memengaruhi statistik seperti:
* rata-rata dan varians,
* kinerja beberapa model Machine Learning.

Oleh karena itu, tergantung pada algoritma yang digunakan, seringkali diperlukan tindakan untuk mengatasi outlier dengan menghapus atau memprosesnya. Ada 2 cara yang umum dipakai untuk handling outlier :
1.   IQR : Interquartile (Pendekatan Statistik)
2.  Nilai Sembarang : (Pendekatan Bisnis)



## 1. Outlier Handling : InterQuartile Handling

In [ ]:
import pandas as pd

In [ ]:
# untuk plot Q-Q
import scipy.stats as stats

In [ ]:
df_california = pd.read_csv('california_dataset.csv')
df_california.head()

# spliting data

In [ ]:
from sklearn.model_selection import train_test_split
train_clfr, test_clfr = train_test_split(df_california, test_size = 0.2, random_state=42)
#NOTES :
#train : test = 80:20 atau 75:25 atau (minimum :70:30 atau maksimum: 90:10) bagi angka yg lain (85:15)
#data train di-handling outliernya
#tapi tidak untuk data test, karena data test ibarat representasi data masa depan yang digunakan untuk melihat performa Machine Learning

In [ ]:
df_california.columns

Disini kita visualisasikan kolom pada data df_california dalam bentuk:
1. histogram
2. plot Q-Q
3. box plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
def check_plot(df, variable):
    # fungsi mengambil kerangka data (df) dan
    # variabel yang diminati sebagai argumen

    # tentukan ukuran gambar
    plt.figure(figsize=(16, 4))

    # histogram
    plt.subplot(1, 3, 1)
    sns.histplot(df[variable], bins=30)
    plt.title('Histogram')

    # plot Q-Q
    plt.subplot(1, 3, 2)
    stats.probplot(df[variable], dist="norm", plot=plt)
    plt.ylabel('Variable quantiles')

    # box plot
    plt.subplot(1, 3, 3)
    sns.boxplot(y=df[variable])
    plt.title('Boxplot')

    plt.show()

Disini saya mengambil contoh pada kolom "Population" untuk divisualisasikan

In [ ]:
check_plot(train_clfr, 'Population')

Dari visualisasi kolom "population" terlihat bahwa ada outlier pada kolom tersebut. Terlihat dari visualisasi yang ditampilkan:
1. Histogram : Sebagian besar data berkumpul di nilai kecil (sebelah kiri), dengan ekor panjang di sisi kanan (skew kanan).
2. Probabilit plot : terlihat titik biru yang berada diluar garis merah, yang menandakan adanya outlier
3. boxplot : Banyak outlier di sisi atas.
Whisker bawah pendek, sedangkan whisker atas panjang.

In [ ]:
def find_outlier_boundary(df, variable):

    # Mari kita hitung batas luar yang merupakan outlier

    IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)

    lower_boundary = df[variable].quantile(0.25) - (IQR * 1.5)
    upper_boundary = df[variable].quantile(0.75) + (IQR * 1.5)

    return upper_boundary, lower_boundary

In [ ]:
Population_upper_limit, Population_lower_limit = find_outlier_boundary(train_clfr, 'Population')
Population_upper_limit, Population_lower_limit



*   Upper Limit (3131.5) → Nilai di atas ini dianggap sebagai outlier.
*   Lower Limit (-616.5) → Nilai di bawah ini juga dianggap sebagai outlier.



In [ ]:
import numpy as np

In [ ]:
# Sekarang mari kita ganti outlier dengan batas maksimum dan minimum

train_clfr['Population'] = np.where(
    train_clfr['Population'] > Population_upper_limit, Population_upper_limit,  # Jika lebih besar dari batas atas, ganti dengan batas atas
    np.where(
        train_clfr['Population'] < Population_lower_limit, Population_lower_limit,  # Jika lebih kecil dari batas bawah, ganti dengan batas bawah
        train_clfr['Population']  # Jika tidak outlier, biarkan tetap sama
    )
)

In [ ]:
# memvisualisasikan outlier
check_plot(train_clfr, 'Population')

Dari distribusi diatas, kita bisa melihat tidak skew kanan yang extreme, melainkan ada 1 nilai di sebelah kanan yang menumpuk akibat handling IQR namun hal ini valid secara statistik. Boxplot pun hilang karena pengaruh IQR ini.

## 2. Outlier Handling : Nilai Sembarang (Business Perspective)

disini kita menggunakan data titanic

In [ ]:
!pip install feature_engine

In [ ]:
from feature_engine.outliers import ArbitraryOutlierCapper

In [ ]:
from sklearn.model_selection import train_test_split
train_titanic, test_titanic = train_test_split(df, test_size = 0.2, random_state=42)
#NOTES :
#train : test = 80:20 atau 75:25 atau (minimum :70:30 atau maksimum: 90:10) bagi angka yg lain (85:15)
#data train di-handling outliernya
#tapi tidak untuk data test, karena data test ibarat representasi data masa depan yang digunakan untuk melihat performa Machine Learning

In [ ]:
df.columns

In [ ]:
train_titanic.age.describe()

Misal : karena alasan keselamatan dan tanggung jawab bersama maka, ada aturan di kapal titanic bahwa anak-anak dibawah 3 tahun dilarang naik kapal, minimal harus 3 tahun dan maksimal umur 80 tahun.

In [ ]:
teknik_capper = ArbitraryOutlierCapper(max_capping_dict={'age': 80},
                                       min_capping_dict={'age': 3})

teknik_capper.fit(train_titanic.fillna(0))
#Dokumentasi : https://feature-engine.trainindata.com/en/1.1.x/outliers/ArbitraryOutlierCapper.html
#asumsikan sudah dilakukan handling missing value dengan teknik sementara yaitu memasukkan nilai 0,
#(nanti nilainya bisa disesuaikan menggunakan median misalnya karena numerikal)
#jika tidak demikian maka .fit akan error karena ada NaN atau nilai null lainnya

In [ ]:
temp_titanic = teknik_capper.transform(train_titanic.fillna(0))

In [ ]:
temp_titanic.age.describe()

Dari describe diatas kita bisa melihat bahwa nilai minimalnya 3 tahun dan maksimalnya 80 tahun.

# **Missing Value Handling**

In [ ]:
df_company = pd.read_csv('company.csv')

In [ ]:
df_company.head()

In [ ]:
df_company.isna().sum()

Apakah benar tidak ada missing value ?

In [ ]:
for column in df_company.columns:
    print(f"============= {column} =================")
    display(df_company[column].value_counts())
    print()

Ternyata terdapat nilai missing value di ketiga kolom itu :
- Rating: -1
- Size: -1, Unknown
- Revenue: -1, Unknown / Non-Applicable


Lalu kita akan cek berapa masing-masing kolom memiliki persentase missing value, dalam statistik ketentuannya ialah **jika diatas 20% maka kita drop kolomnya**, **jika <= 20% maka di handling karena alasan distribusi data**.

### Aturan missing value handling :
1. jika tipe datanya itu **numerik** -> handling menggunakan **median** (robust / tahan terhadap outlier)
2. jika tipe datanya itu **kategorik / object / string** -> handling menggunakan **mode / modus**

In [ ]:
100 * len(df_company[df_company['Rating']==-1]) / len(df_company) #untuk menghitung persentase data dengan nilai Rating = -1 dalam dataset

In [ ]:
100 * len(df_company[df_company['Size'].isin(['-1','Unknown'])]) / len(df_company) #Menghitung persentase data yang kolom Size-nya bernilai '-1' atau 'Unknown'

In [ ]:
100 * len(df_company[df_company['Revenue'].isin(['-1','Unknown / Non-Applicable'])]) / len(df_company)

Kita akan drop kolom Revenue karena diatas 20% nilai missing kolomnya.

In [ ]:
df_company = df_company.drop(columns=['Revenue'])
df_company.info()

Untuk kolom Rating disini kita akan melakukan handling dengan menggunakan **median**

In [ ]:
from sklearn.model_selection import train_test_split
train_company, test_company = train_test_split(df_company, test_size = 0.2, random_state=42)

In [ ]:
# Imputasi median pada kolom Rating : train

median_rating_train = train_company[train_company['Rating']!=-1]['Rating'].median()

Mencari nilai **median** dari kolom **Rating** tanpa memasukkan nilai -1 (karena -1 digunakan sebagai penanda nilai yang hilang).
Hasilnya disimpan dalam **median_rating_train**, yang akan digunakan untuk menggantikan nilai -1.

In [ ]:
median_rating_train

In [ ]:
#menerapkan imputasi nilai yang hilang untuk melatih dan menguji, dari pemeringkatan nilai median dalam data train
train_company['Rating'] = train_company['Rating'].apply(lambda x: median_rating_train if x==-1 else x) #train
test_company['Rating'] = test_company['Rating'].apply(lambda x: median_rating_train if x==-1 else x) #test

Mengganti semua nilai -1 dalam Rating dengan nilai median_rating_train baik untuk train maupun test.

In [ ]:
train_company['Rating'].value_counts()

* Setelah imputasi, nilai -1 sudah tidak ada lagi di train_company['Rating'].
* Median dari train kan 3.8, sehingga semua nilai -1 diubah menjadi 3.8.
* Karena ada banyak nilai -1 yang diubah menjadi 3.8, maka frekuensi 3.8 meningkat drastis menjadi 72.
* Nilai-nilai lain tetap memiliki distribusi aslinya karena tidak berubah.

# **Encoding**

Untuk membaca jenis data dari tiap kolom kita bisa panggil dengan df_churn.info()

In [ ]:
df_churn = pd.read_csv('TelcoCustomerChurn.csv')
df_churn.head(6)

In [ ]:
# Mengganti "Yes" menjadi 1 dan "No" menjadi 0 dalam kolom "Churn"
df_churn['Churn'] = df_churn['Churn'].replace({'Yes': 1, 'No': 0})
df_churn.head()

In [ ]:
#Cek nilai-nilai di kolom kategorikal
custom_columns = ['gender','Partner','Dependents','PhoneService','MultipleLines',
                  'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection'
                  ,'TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Churn']
for column in custom_columns:
    print(f"============= {column} =================")
    display(df_churn[column].value_counts())
    print()

Jika teman-teman lihat, ada beberapa kolom memiliki nilai 'No internet service' atau 'No phone service', yang artinya pelanggan tidak memiliki layanan tersebut.
Untuk menyederhanakan data, nilai 'No internet service' diubah menjadi 'No' dalam fitur layanan internet.

In [ ]:
df_churn['StreamingMovies'] = df_churn['StreamingMovies'].replace({'No internet service':'No'})
df_churn['StreamingTV'] = df_churn['StreamingTV'].replace({'No internet service':'No'})
df_churn['TechSupport'] = df_churn['TechSupport'].replace({'No internet service':'No'})
df_churn['DeviceProtection'] = df_churn['DeviceProtection'].replace({'No internet service':'No'})
df_churn['OnlineBackup'] = df_churn['OnlineBackup'].replace({'No internet service':'No'})
df_churn['OnlineSecurity'] = df_churn['OnlineSecurity'].replace({'No internet service':'No'})
df_churn['MultipleLines'] = df_churn['MultipleLines'].replace({'No phone service':'No'})

In [ ]:
#Cek nilai-nilai di kolom kategorikal
for column in custom_columns:
    print(f"============= {column} =================")
    display(df_churn[column].value_counts())
    print()

## **One Hot Encoding**

In [ ]:
train_churn, test_churn = train_test_split(df_churn, test_size = 0.2, random_state = 42)

In [ ]:
# Melakukan one-hot encoding pada kolom "gender"
# Kolom-kolom yang ingin di one-hot encoding
encode_ohe = ['gender'] #buat list nama-nama kolom yang ingin dilakukan OHE
train_churn = pd.get_dummies(train_churn, columns=encode_ohe, dtype=int)
test_churn = pd.get_dummies(test_churn, columns=encode_ohe, dtype=int)

In [ ]:
train_churn.head()

In [ ]:
test_churn.head()

## **Label Encoder**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Kolom-kolom yang ingin di-label encode
columns_to_encode = ['Partner', 'Dependents']

# Inisialisasi LabelEncoder
label_encoder = LabelEncoder()

In [ ]:
# Menerapkan label encoding ke data pelatihan (train)
for col in columns_to_encode:
    train_churn[col] = label_encoder.fit_transform(train_churn[col])
#fit dilakukan di data train dan implementasikan ke train, kemudian lakukan transform ke data test Untuk mencegah data leakage

In [ ]:
# Menerapkan label encoding yang sama ke data uji (test)
for col in columns_to_encode:
    test_churn[col] = label_encoder.transform(test_churn[col])

In [ ]:
train_churn.head()

In [ ]:
test_churn.head()

# **Feature Scalling**

Proses perubahan nilai skala dengan pendekatan statistik, dengan mengubah nilai tapi tidak mengubah makna. Tekniknya :

1. StandarScaller
2. MinMaxScaller

Notes :
Pada pembahasan ini, akan dilakukan pada kolom yang sama bertujuan untuk melakukan cara penggunaan libraries nya saja. Pada kolom : ['tenure', 'MonthlyCharges']

## 1. StandarScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

# Kolom-kolom yang ingin di-standarisasi
columns_to_stdscaller = ['tenure', 'MonthlyCharges']

# Inisialisasi StandardScaler
scaler = StandardScaler()

In [ ]:
train_churn.info()

In [ ]:
# Menerapkan standarisasi ke data pelatihan (train)
# Convert 'tenure' and 'MonthlyCharges' to numeric if they are datetime objects
train_churn['tenure'] = pd.to_numeric(train_churn['tenure'], errors='coerce')
train_churn['MonthlyCharges'] = pd.to_numeric(train_churn['MonthlyCharges'], errors='coerce')
train_churn[['tenure_stds', 'MonthlyCharges_stds']] = scaler.fit_transform(train_churn[columns_to_stdscaller])

# Menerapkan standarisasi yang sama ke data uji (test)
# Convert 'tenure' and 'MonthlyCharges' to numeric if they are datetime objects
test_churn['tenure'] = pd.to_numeric(test_churn['tenure'], errors='coerce')
test_churn['MonthlyCharges'] = pd.to_numeric(test_churn['MonthlyCharges'], errors='coerce')
test_churn[['tenure_stds', 'MonthlyCharges_stds']] = scaler.transform(test_churn[columns_to_stdscaller])

In [ ]:
# kita lihat distribusi dari variabel sebelum standar scaller dan setelah standar scaller

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# Plot distribusi sebelum Standar Scaler
ax1.set_title('Sebelum Standar Scaller')
sns.kdeplot(train_churn['tenure'], ax=ax1, label='Tenure')
sns.kdeplot(train_churn['MonthlyCharges'], ax=ax1, label='Monthly Charges')

# Plot distribusi setelah Standar Scaler
ax2.set_title('Setelah Standard Scaller')
sns.kdeplot(train_churn['tenure_stds'], ax=ax2, label='Tenure (scaled)')
sns.kdeplot(train_churn['MonthlyCharges_stds'], ax=ax2, label='Monthly Charges (scaled)')

# Set x-labels
ax1.set_xlabel('Rentang Nilai Sebelum Feature Scalling (Standar Scaller)')
ax2.set_xlabel('Rentang Nilai Setelah Feature Scalling (Standar Scaller)')

# Menampilkan legenda
ax1.legend()
ax2.legend()

plt.show()

## 2. MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Kolom-kolom yang ingin dinormalisasi
columns_to_normalize = ['tenure', 'MonthlyCharges']

# Inisialisasi MinMaxScaler
minmax = MinMaxScaler()

In [ ]:
# Menerapkan normalisasi Min-Max ke data pelatihan (train)
train_churn[['tenure_minmax', 'MonthlyCharges_minmax']] = minmax.fit_transform(train_churn[columns_to_normalize])

# Menerapkan normalisasi Min-Max yang sama ke data uji (test)
test_churn[['tenure_minmax', 'MonthlyCharges_minmax']] = minmax.transform(test_churn[columns_to_normalize])

In [ ]:
# Plot distribusi sebelum dan sesudah Min-Max Scaling
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# Plot distribusi sebelum Min-Max Scaling
ax1.set_title('Sebelum Min-Max Scaling')
sns.kdeplot(train_churn['tenure'], ax=ax1, label='tenure')
sns.kdeplot(train_churn['MonthlyCharges'], ax=ax1, label='MonthlyCharges')

# Plot distribusi setelah Min-Max Scaling
ax2.set_title('Setelah Min-Max Scaling')
sns.kdeplot(train_churn['tenure_minmax'], ax=ax2, label='tenure (scaled)')
sns.kdeplot(train_churn['MonthlyCharges_minmax'], ax=ax2, label='MonthlyCharges (scaled)')

# Set x-labels
ax1.set_xlabel('Rentang Nilai Sebelum Min-Max Scaling')
ax2.set_xlabel('Rentang Nilai Setelah Min-Max Scaling')

# Menampilkan legenda
ax1.legend()
ax2.legend()

plt.show()

In [ ]:
train_churn[['tenure_minmax','MonthlyCharges_minmax']].describe()

## Selesai